In [ ]:
def simulate_generations(
    initial_poAncestry_1: list = None,
    initial_poAncestry_2: list = None,
    generation_plan: list = None,
    num_offspring_per_cross: int = 2, # This parameter determines family size for each mating pair.
    num_chromosomes: int = 2,
    recomb_event_probabilities: dict = None,
    recomb_probabilities: list = None,
    existing_populations: dict = None,
    verbose: bool = False,
):
    """
    Orchestrates the simulation of multiple genetic generations based on a predefined crossing plan.

    This function manages the creation, crossing, and data recording of individuals
    across different generations (e.g., F1, F2, Backcrosses). It takes initial populations,
    a detailed plan of crosses, and genetic parameters to simulate inheritance and recombination.
    It returns all generated population data and aggregated genetic statistics.

    Args:
        initial_poAncestry_1 (list, optional): The initial population of Ancestry 1 individuals. Defaults to None.
        initial_poAncestry_2 (list, optional): The initial population of Ancestry 2 individuals. Defaults to None.
        generation_plan (list, optional): A list of tuples defining the crosses for each generation.
                                          Each tuple is (new_gen_label, parent1_label, parent2_label).
                                          Defaults to None.
        num_offspring_per_cross (int, optional): The number of offspring generated from each
                                                 unique mating pair in a cross. Defaults to 2.
        num_chromosomes (int, optional): The number of diploid chromosome pairs for offspring. Defaults to 2.
        recomb_event_probabilities (dict, optional): Probabilities for 0, 1, 2 recombination events per chromosome.
                                                     Defaults to None.
        recomb_probabilities (list, optional): Position-dependent probabilities for recombination between loci.
                                               Defaults to None.
        existing_populations (dict, optional): A dictionary of pre-existing populations to start from.
                                               Keys are population labels, values are lists of Individuals.
                                               Defaults to None.
        verbose (bool, optional): If True, prints detailed statistics and messages during simulation.
                                  Defaults to False.

    Returns:
        Tuple[dict, dict, list, list]: A tuple containing:
            - populations (dict): A dictionary where keys are generation labels (str) and values are
                                  lists of 'Individual' objects for that generation.
            - all_generations_data (dict): A dictionary where keys are generation labels (str) and values
                                           are lists of dictionaries, each containing HI/HET for individuals.
            - all_locus_genotype_data (list): A global list containing detailed genotype data for every locus.
            - all_chromatid_recombination_data (list): A global list containing detailed recombination block data.
    """
    # Initialize the dictionary to store all populations generated throughout the simulation.
    # If `existing_populations` are provided, they are used as the starting point.
    populations = existing_populations if existing_populations is not None else {}

    # Initialize a dictionary to store per-individual HI and HET data for each generation.
    all_generations_data = {}

    # If initial Ancestry 1 population is provided and not already in `populations`, add it.
    if initial_poAncestry_1 is not None and 'Ancestry_1' not in populations:
        populations['Ancestry_1'] = initial_poAncestry_1
        # Record the full genome and recombination data for all individuals in this initial population.
        for ind in initial_poAncestry_1:
            record_individual_genome(ind, 'Ancestry_1')
            record_chromatid_recombination(ind, 'Ancestry_1')
        # Calculate and store the HI/HET for individuals in Ancestry 1.
        all_generations_data['Ancestry_1'] = calculate_hi_het_for_population(initial_poAncestry_1)

    # If initial Ancestry 2 population is provided and not already in `populations`, add it.
    if initial_poAncestry_2 is not None and 'Ancestry_2' not in populations:
        populations['Ancestry_2'] = initial_poAncestry_2
        # Record the full genome and recombination data for all individuals in this initial population.
        for ind in initial_poAncestry_2:
            record_individual_genome(ind, 'Ancestry_2')
            record_chromatid_recombination(ind, 'Ancestry_2')
        # Calculate and store the HI/HET for individuals in Ancestry 2.
        all_generations_data['Ancestry_2'] = calculate_hi_het_for_population(initial_poAncestry_2)

    # If no generation plan is provided, return the current state of populations and data.
    if generation_plan is None:
        print("Warning: No generation plan provided. Returning existing populations.")
        return populations, all_generations_data, all_locus_genotype_data, all_chromatid_recombination_data

    # Main loop: Iterate over each planned cross in the `generation_plan`.
    for gen_info in generation_plan:
        # Skip if the plan entry only contains a generation label without cross information.
        if len(gen_info) == 1:
            continue

        gen_name = gen_info[0]       # The label for the new generation (e.g., 'F1', 'BC1_A').
        ancestors_names = gen_info[1:] # Labels for the parental populations for this cross.

        # Validate that the specified parental populations exist in the `populations` dictionary.
        for p_name in ancestors_names:
            if p_name not in populations:
                raise ValueError(f"Parental population '{p_name}' not found for generation '{gen_name}'. "
                                 f"Available populations: {list(populations.keys())}")

        # Retrieve the actual lists of Individual objects for the two parental populations.
        ancestors_poAncestry_1_for_cross = populations[ancestors_names[0]]
        ancestors_poAncestry_2_for_cross = populations[ancestors_names[1]]

        # Execute the genetic cross using the `run_genetic_cross` function.
        # This will generate the new offspring population.
        new_pop = run_genetic_cross(
            ancestors_poAncestry_1_for_cross,
            ancestors_poAncestry_2_for_cross,
            offspring_count_per_mating_pair=num_offspring_per_cross,
            generation_label=gen_name,
            num_chromosomes_for_offspring=num_chromosomes,
            recomb_event_probabilities=recomb_event_probabilities,
            recomb_probabilities=recomb_probabilities
        )

        # Store the newly generated population in the `populations` dictionary.
        populations[gen_name] = new_pop

        # Calculate and store the HI/HET data for all individuals in this new generation.
        all_generations_data[gen_name] = calculate_hi_het_for_population(new_pop)

        # Print debug message indicating the size of the new population.
        print(f"DEBUG: Generated population {gen_name} with {len(new_pop)} individuals.")

        # If verbose mode is enabled, print more detailed statistics for the new generation.
        if verbose:
            stats = population_stats(new_pop) # Get summary statistics for the new population.
            print(f"{gen_name} created from parents {ancestors_names[0]} and {ancestors_names[1]} | "
                  f"Count: {len(new_pop)} | Mean HI: {stats['mean_HI']:.3f} (±{stats['std_HI']:.3f}), "
                  f"Mean HET: {stats['mean_HET']:.3f} (±{stats['std_HET']:.3f})")
            print(f"Added '{gen_name}' to populations. Current population keys: {list(populations.keys())}")

    # After simulating all generations in the plan, return all collected data.
    # Note: `all_locus_genotype_data` and `all_chromatid_recombination_data` are assumed
    # to be global lists that the `record_...` functions append to.
    return populations, all_generations_data, all_locus_genotype_data, all_chromatid_recombination_data